This script is respponsible for training the Models on the given dataset and save the best metadata.

1. Set Up Environment

In [1]:
# Install required packages
!pip install aim pyngrok albumentations schedulefree torchinfo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 54.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 8.1 MB/s eta 0:00:00
  Created wheel for aim-ui: filename=aim_ui-3.29.1-py3-none-any.whl size=31195131 sha256=d282e75c319b8560da886f164d1aeff348f410e31163b272c2736272895c6ceb
  Stored in directory: /root/.cache/pip/wheels/66/52/46/97539bf69f8ac8071748157cd835404e6e6cd7396b809b1782
Successfully built aim-ui


In [2]:
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

  Cloning https://github.com/qubvel/segmentation_models.pytorch to /tmp/pip-req-build-1k50fbqy
  Running command git clone --filter=blob:none --quiet https://github.com/qubvel/segmentation_models.pytorch /tmp/pip-req-build-1k50fbqy
  Resolved https://github.com/qubvel/segmentation_models.pytorch to commit 4d20629756005085ca0f1f21605c796298ca0c16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for segmentation_models_pytorch: filename=segmentation_models_pytorch-0.5.1.dev0-py3-none-any.whl size=155806 sha256=df875679b6d2226370278c2a772bb5810e34b422de60dc5a123cfb1f341e8a31
  Stored in directory: /tmp/pip-ephem-wheel-cache-r_zm4wls/wheels/ef/38/4b/267c9bdb27c85ebaa11e9ec77c9059cf2c166ceee760f24429
Successfully built segmentation_models_pytorch


# --- 1. Imports ---

In [3]:
# --- 1. Imports ---
print("Importing libraries...")
from google.colab import drive
drive.mount('/content/drive')

import schedulefree
import zipfile
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
from google.colab import userdata
import random
import seaborn as sns
import shutil
import smtplib
from email.mime.text import MIMEText
from torch.cuda.amp import autocast, GradScaler
import cv2
import segmentation_models_pytorch as smp
from torchinfo import summary
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import warnings
from sklearn.metrics import auc as sklearn_auc
import json

#warnings.filterwarnings("ignore")
from sklearn.metrics import roc_auc_score
from torchvision import transforms
from torchvision.transforms import functional as TF
import timm  # For Swin Transformer
from sklearn.metrics import confusion_matrix, accuracy_score
from PIL import Image, UnidentifiedImageError
import torchvision.models as models

print("Libraries imported.")

Importing libraries...
Mounted at /content/drive
Libraries imported.


In [4]:
# --- 5. Configuration & Setup ---
print("Configuring environment...")
# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Hyperparameters ---
BATCH_SIZE = 16 # Slightly reduced default batch size, adjust based on GPU memory
NUM_EPOCHS = 100
WORKERS = 2 # Keep > 0 for persistent workers
PATIENCE = 5 # Increased patience slightly
FREEZE_EPOCHS = 0 # Number of epochs to keep encoder frozen
BY_PASS_NGROK = True # Set to False to enable ngrok/Aim UI tunnel
UNLEASHED = False # Set to True to disable early stopping

# --- Add SMP specific config ---
ENCODER = "inceptionresnetv2"  # Choose your desired encoder from SMP list
ENCODER_WEIGHTS = "imagenet" # Use pre-trained weights
ACTIVATION = None # Output raw logits for BCEWithLogitsLoss/Hybrid loss

# --- Paths and Labels ---
CHECKPOINT_PATH = None # Can be set to resume training

# --- Email Settings ---
sender = "bruno.nunes.1987@gmail.com"
recipients = ["bruno.nunes.1987@gmail.com"]
password = userdata.get('APP_PASSWORD') # Ensure this secret exists in Colab

# --- Aim Repository Path ---
AIM_REPO_PATH = "/content/drive/MyDrive/aim_repo_prostate"

# --- Dataset Source Path ---
# Assumes zip files named like 'fold_1.zip', 'fold_2.zip', etc. exist here
# Contains pre-processed, balanced data from script 4
DATASET_ZIP_DIR = '/content/drive/MyDrive/IA_MEDICA_SAMPLES/ENSEMBLE_CLEAN'

# --- Data Extraction Directory ---
base_data_dir = '/content/dataset' # Extracted fold data goes here

# --- Paths within the extracted fold ---
# These are consistent relative to base_data_dir after extraction
train_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/CANCER')
train_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/CANCER_MASK')
train_not_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER')
train_not_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER_MASK')
val_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER')
val_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER_MASK')
val_not_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER')
val_not_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER_MASK')

# --- Loss Weights (from paper) ---
ALPHA_BCE = 0.5
BETA_DICE_BG = 0.25
GAMMA_DICE_FG = 0.25

#replication
SEED = 42

# Use a consistent, reasonable LR unless you have strong fold-specific reasons
BASE_LEARNING_RATE = 1e-5 # Define the main learning rate here
ENCODER_LR_FACTOR = 0.1 # Encoder LR will be BASE_LEARNING_RATE * ENCODER_LR_FACTOR
WEIGHT_DECAY = 1e-5 # Regularization strength
DECODER_DROPOUT = 0.5 # Define your desired dropout rate

Configuring environment...
Using device: cuda


In [5]:
# --- 2. Dice Loss Implementation ---
def dice_coeff_per_class(pred_prob, target_one_hot, class_index, smooth=1e-6):
    """
    Calculates the Dice Coefficient for a specific class.
    Args:
        pred_prob (torch.Tensor): Predicted probabilities (softmax output). Shape: [B, C, H, W]
        target_one_hot (torch.Tensor): Ground truth labels (one-hot encoded). Shape: [B, C, H, W]
        class_index (int): The index of the class to calculate Dice for.
        smooth (float): Smoothing factor to avoid division by zero.
    Returns:
        torch.Tensor: Dice coefficient for the specified class (scalar averaged over batch).
    """
    pred_c = pred_prob[:, class_index, :, :]    # Shape: [B, H, W]
    target_c = target_one_hot[:, class_index, :, :] # Shape: [B, H, W]

    intersection = torch.sum(pred_c * target_c, dim=(1, 2)) # Shape: [B]
    sum_pred = torch.sum(pred_c, dim=(1, 2))                # Shape: [B]
    sum_target = torch.sum(target_c, dim=(1, 2))            # Shape: [B]

    dice = (2. * intersection + smooth) / (sum_pred + sum_target + smooth) # Shape: [B]

    # Average Dice coefficient over the batch
    return dice.mean()

def dice_loss_per_class(pred_prob, target_one_hot, class_index, smooth=1e-6):
    """Calculates Dice Loss (1 - Dice Coefficient) for a specific class."""
    dice_coefficient = dice_coeff_per_class(pred_prob, target_one_hot, class_index, smooth)
    return 1.0 - dice_coefficient


# --- 3. Combined BCE + Dice Loss Function ---
def bce_dice_hybrid_loss(pred_logits, target_one_hot,
                          alpha=ALPHA_BCE, beta=BETA_DICE_BG, gamma=GAMMA_DICE_FG, # Weights from paper
                          smooth=1e-6):
    """
    Combined BCE + Dice(BG) + Dice(FG) loss.

    Args:
        pred_logits (torch.Tensor): Model predictions (logits). Shape: [B, C, H, W]
        target_one_hot (torch.Tensor): Ground truth labels (one-hot encoded). Shape: [B, C, H, W]
        alpha (float): Weight for BCE loss.
        beta (float): Weight for Background Dice loss (class 0).
        gamma (float): Weight for Foreground Dice loss (class 1).
        smooth (float): Smoothing factor for Dice calculation.

    Returns:
        torch.Tensor: Combined loss value.
    """
    # 1. BCE Loss (using logits for stability)
    # Ensure target is float for BCEWithLogitsLoss
    bce_loss = F.binary_cross_entropy_with_logits(pred_logits, target_one_hot.float(), reduction='mean')

    # 2. Softmax probabilities for Dice Loss calculation
    pred_prob = torch.softmax(pred_logits, dim=1)

    # 3. Dice Loss for Background (Class 0)
    dice_loss_bg = dice_loss_per_class(pred_prob, target_one_hot, class_index=0, smooth=smooth)

    # 4. Dice Loss for Foreground (Class 1 - Cancer)
    dice_loss_fg = dice_loss_per_class(pred_prob, target_one_hot, class_index=1, smooth=smooth)

    # 5. Weighted combination based on paper's formula (Eq. 3)
    combined_loss = (alpha * bce_loss +
                     beta * dice_loss_bg +
                     gamma * dice_loss_fg)

    return combined_loss

In [6]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [7]:
# --- Seeding ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == torch.device('cuda'):
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Keep benchmark=False for determinism if needed, but True might be faster
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

Configuration complete.


# ***Define METRICS FOR EVALUATION***

In [8]:
# --- 6. Metrics Functions (Keep as is, checked previously) ---
print("Defining metric functions...")
def iou_coefficient(pred, target, smooth=1e-6):
    # ... (iou_coefficient code - seems robust) ...
    pred = pred.squeeze(1).float()
    target = target.squeeze(1).float()
    intersection = torch.sum(pred * target, dim=(1, 2))
    sum_pred = torch.sum(pred, dim=(1, 2))
    sum_target = torch.sum(target, dim=(1, 2))
    union = sum_pred + sum_target - intersection
    iou_val = torch.where(union > smooth, (intersection + smooth) / (union + smooth), torch.ones_like(intersection))
    iou_val = torch.clamp(iou_val, 0.0, 1.0)
    if torch.any(torch.isnan(iou_val)) or torch.any(torch.isinf(iou_val)): iou_val = torch.nan_to_num(iou_val, nan=0.0, posinf=0.0, neginf=0.0)
    result = iou_val.mean()
    if torch.isnan(result) or torch.isinf(result): return 0.0
    return torch.clamp(result, 0.0, 1.0).item()


def dice_coefficient(pred, target, smooth=1e-6):
    # ... (dice_coefficient code - seems robust) ...
    pred = pred.squeeze(1).float()
    target = target.squeeze(1).float()
    intersection = torch.sum(pred * target, dim=(1, 2))
    sum_pred = torch.sum(pred, dim=(1, 2))
    sum_target = torch.sum(target, dim=(1, 2))
    denominator = sum_pred + sum_target
    dice = torch.where(denominator > smooth, (2. * intersection + smooth) / (denominator + smooth), torch.ones_like(intersection))
    dice = torch.clamp(dice, 0.0, 1.0)
    if torch.any(torch.isnan(dice)) or torch.any(torch.isinf(dice)): dice = torch.nan_to_num(dice, nan=0.0, posinf=0.0, neginf=0.0)
    result = dice.mean()
    if torch.isnan(result) or torch.isinf(result): return 0.0
    return torch.clamp(result, 0.0, 1.0).item()

print("Metric functions defined.")

Defining metric functions...
Metric functions defined.


4. Create Custom Dataset

In [9]:
# --- 8. Dataset Class (Simplified) ---
print("Defining simplified ProstateCancerDataset...")
# Use albumentations for basic transforms (Resize, Normalize, ToTensor)
class ProstateCancerDataset(Dataset):
    def __init__(self, cancer_image_dir, cancer_mask_dir, not_cancer_image_dir, not_cancer_mask_dir):
        # Removed is_train flag as augmentations are pre-applied
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        # --- Base Transformation (Applied to ALL data) ---
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR), # Specify interpolation
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(), # Handles image normalization (scaling) and channel order (C, H, W)
                          # Converts mask to Tensor (C, H, W)
        ])

        # --- Load File Lists ---
        # Defensive listing: check if dirs exist
        self.cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
             self.cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.cancer_image_dir}")

        self.not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
             self.not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.not_cancer_image_dir}")


        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = [] # Still useful maybe for checks later

        for img_name in self.cancer_images:
             img_path = os.path.join(self.cancer_image_dir, img_name)
             mask_path = os.path.join(self.cancer_mask_dir, img_name)
             if os.path.isfile(mask_path): # Ensure mask exists
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(1)
             # else: print(f"Warning: Mask missing for cancer image {img_name}")

        for img_name in self.not_cancer_images:
             img_path = os.path.join(self.not_cancer_image_dir, img_name)
             mask_path = os.path.join(self.not_cancer_mask_dir, img_name)
             if os.path.isfile(mask_path): # Ensure mask exists
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(0)
             # else: print(f"Warning: Mask missing for non-cancer image {img_name}")

    def __len__(self):
        # Length is simply the total number of valid image/mask pairs found
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        try:
            # Load image using OpenCV (as Albumentations often uses it) - loads BGR
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None # Return None tuple on transform error


        return final_image, final_mask

print("Dataset definition complete.")

Defining simplified ProstateCancerDataset...
Dataset definition complete.


# ***Early Stop Class***

In [10]:
# --- 9. Early Stopping Class (Modified for Min Val Loss) ---
print("Defining EarlyStopping class (monitoring Validation Loss)...")
class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0.0001, mode='min'):
        """
        Args:
            patience (int): How long to wait after last time validation metric improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation metric improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                           For mode='min', it's best_score - current_score > delta.
                           For mode='max', it's current_score - best_score > delta.
                           Default: 0.0001
            mode (str): One of 'min' or 'max'. In 'min' mode, training stops when the quantity
                        monitored stops decreasing; in 'max' mode it stops when the quantity
                        monitored stops increasing. Default: 'min'
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.early_stop = False
        self.best_score = None
        self.val_loss_min = np.inf # Explicitly track min loss
        self.delta = delta
        self.mode = mode.lower()
        self.previous_best_score = None # Store previous best for logging

        if self.mode not in ['min', 'max']:
            raise ValueError("mode has to be 'min' or 'max'")

        if self.mode == 'min':
            self.delta *= -1 # For comparison: score < best_score + (-delta)

    def __call__(self, score, model, path, checkpoint_path, optimizer, epoch, val_loss, val_iou):
        """
        Args:
            score (float): The score to monitor (e.g., validation loss).
            model (torch.nn.Module): Model to save.
            path (str): Path to save the best model checkpoint.
            checkpoint_path (str): Path to save the periodic checkpoint.
            optimizer: Optimizer state to save.
            epoch (int): Current epoch number.
            val_loss (float): Current validation loss (for saving in checkpoint).
            val_iou (float): Current validation IoU (for saving in checkpoint).
        """

        # Initialize best_score on the first call
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path, checkpoint_path, optimizer, epoch, val_iou, score)
            return

        # Check for improvement based on mode
        improvement_detected = False
        if self.mode == 'min':
            if score < self.best_score + self.delta:
                 improvement_detected = True
        else:
            if score > self.best_score + self.delta:
                 improvement_detected = True

        if improvement_detected:
            self.previous_best_score = self.best_score # Store previous best
            self.best_score = score
            self.save_checkpoint(val_loss, model, path, checkpoint_path, optimizer, epoch, val_iou, score) # Pass score
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience} (Best {self.mode} score: {self.best_score:.6f})')
            if self.counter >= self.patience:
                self.early_stop = True
        return improvement_detected

    def save_checkpoint(self, val_loss, model, path, checkpoint_path, optimizer, epoch, val_iou, score):
        '''Saves model when monitored score improves.'''
        if self.verbose:
            if self.previous_best_score is None:
                 print(f'Initial best score ({self.mode} mode): {score:.6f}. Saving model ...')
            else:
                 direction = "decreased" if self.mode == 'min' else "increased"
                 print(f'Validation score {direction} ({self.previous_best_score:.6f} --> {score:.6f}). Saving model ...')

        # --- Saving logic remains the same ---
        is_compiled = hasattr(model, '_orig_mod')
        model_to_save = model._orig_mod if is_compiled else model
        model_to_save.cpu()
        model.eval() # Don't put model in eval mode if optimizer state is saved

        save_dict = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss, # Still save actual val_loss
            'val_iou': val_iou,   # Still save actual val_iou
            'best_val_score': score, # Save the score that triggered the save
            'is_compiled': is_compiled,
        }
        try:
            torch.save(save_dict, path)
            if checkpoint_path and path != checkpoint_path:
                 torch.save(save_dict, checkpoint_path)
        except Exception as e:
             print(f"Error saving checkpoint: {e}")
        finally:
          model_to_save.to(device) # Move back to original device

print("EarlyStopping class defined (monitoring Validation Loss).")

Defining EarlyStopping class (monitoring Validation Loss)...
EarlyStopping class defined (monitoring Validation Loss).


In [11]:
# --- 10. Training & Validation Functions (Corrected Loss Input) ---
print("Defining training and validation functions...")
# --- TRAINING FUNCTION ---
def train_model(model, optimizer, dataloader, device, current_epoch, freeze_epochs):
    model.train()
    # --- Freezing Logic using SMP structure ---
    is_frozen_now = current_epoch < freeze_epochs
    if is_frozen_now and getattr(model, '_encoder_frozen', True):
        # Freeze encoder parameters if not already done
        print(f"Epoch {current_epoch+1}/{NUM_EPOCHS}: Keeping encoder frozen.")
        for param in model.encoder.parameters(): # Target SMP encoder
            param.requires_grad = False
        model._encoder_frozen = True
    elif not is_frozen_now and getattr(model, '_encoder_frozen', True):
        # Unfreeze encoder parameters
        print(f"Epoch {current_epoch+1}/{NUM_EPOCHS}: Unfreezing encoder.")
        for param in model.encoder.parameters(): # Target SMP encoder
            param.requires_grad = True
        model._encoder_frozen = False
    # --- End Freezing Logic ---

    optimizer.train() # Make sure optimizer is aware of mode change (though schedulefree might handle it)

    running_loss=0.0;
    running_iou=0.0;
    running_dice=0.0;
    num_samples_processed=0;
    num_batches_processed=0;
    nan_flag=False
    optimizer.zero_grad(set_to_none=True);
    scaler = torch.amp.GradScaler('cuda')

    pbar=tqdm(dataloader, desc=f"Train E{current_epoch+1}", leave=False)
    for batch_data in pbar:
        if batch_data is None:
          print("Warning: Skipping None batch yielded by DataLoader.")
          continue;

        images, masks = batch_data;

        # 3. Check if unpacking resulted in None (should be unlikely with collate_fn, but defensive)
        if images is None or masks is None:
            print("Warning: Skipping batch with None images or masks after unpacking.")
            continue

        current_batch_size = images.size(0)

        # 5. Check for empty batch
        if current_batch_size == 0:
            print("Warning: Skipping batch with size 0.")
            continue

        images=images.to(device,non_blocking=True);
        masks=masks.to(device,non_blocking=True).float()

        with torch.amp.autocast('cuda'):
          outputs_raw = model(images)

          # --- Handle potential tuple output from compiled model ---
          if isinstance(outputs_raw, tuple):
              if not outputs_raw: # Empty tuple
                  print("Warning: Compiled model returned empty tuple. Skipping.")
                  continue
              outputs = outputs_raw[0] # Assume first element is the main output
          else:
              outputs = outputs_raw # Assume it's the tensor directly
          # --- End tuple handling ---

                    # Check if outputs is a tensor before accessing shape
          if not isinstance(outputs, torch.Tensor):
                print(f"Warning: Model output after potential tuple handling is not a tensor ({type(outputs)}). Skipping.")
                continue

          if outputs.shape != masks.shape:
             print(f"Warning: Shape mismatch! Output: {outputs.shape}, Mask: {masks.shape}. Skipping batch.")
             continue # Skip if shapes mismatch

          loss=bce_dice_hybrid_loss(outputs, masks, ALPHA_BCE, BETA_DICE_BG, GAMMA_DICE_FG)

        if torch.isnan(loss) or torch.isinf(loss):
          nan_flag=True;
          print(f"\nNaN/Inf Loss!");
          optimizer.zero_grad(set_to_none=True);
          continue

        scaler.scale(loss).backward();
        scaler.unscale_(optimizer);
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);
        scaler.step(optimizer);
        scaler.update();
        optimizer.zero_grad(set_to_none=True)

        # Metrics
        batch_loss=loss.item();
        running_loss+=batch_loss*current_batch_size;
        num_samples_processed+=current_batch_size

        with torch.no_grad():
          predicted_class=torch.argmax(torch.softmax(outputs,1),1).unsqueeze(1);
          true_classes=torch.argmax(masks,1).unsqueeze(1);
          batch_mean_iou=iou_coefficient(predicted_class,true_classes);
          batch_mean_dice=dice_coefficient(predicted_class,true_classes);
          running_iou+=batch_mean_iou;
          running_dice+=batch_mean_dice;
          num_batches_processed+=1

        if num_batches_processed >0 and num_samples_processed>0:
          pbar.set_postfix(loss=f'{batch_loss:.4f}',avgL=f'{running_loss/num_samples_processed:.4f}',avgI=f'{running_iou/num_batches_processed:.4f}',avgD=f'{running_dice/num_batches_processed:.4f}')

    if num_samples_processed==0 or num_batches_processed==0:
       return 0.0,0.0,0.0

    epoch_loss = running_loss / num_samples_processed
    epoch_iou = running_iou / num_batches_processed
    epoch_dice = running_dice / num_batches_processed

    if nan_flag: print("\nWarn: NaN/Inf during training.")

    return epoch_loss, epoch_iou, epoch_dice

Defining training and validation functions...


In [12]:
# --- VALIDATION FUNCTION ---
def validate_model(model, optimizer, dataloader, device):
    model.eval(); optimizer.eval() # Set optimizer to eval mode too
    running_loss = 0.0; running_iou = 0.0; running_dice = 0.0
    num_samples_processed = 0; num_batches_processed = 0
    nan_detected = False

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validate", leave=False)
        for batch_idx, batch_data in enumerate(pbar):
            if batch_data is None: continue
            images, masks = batch_data
            if images is None or masks is None: continue
            current_batch_size = images.size(0)
            if current_batch_size == 0: continue

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            with torch.amp.autocast('cuda'):
                outputs_raw = model(images)

                # --- Handle potential tuple output from compiled model ---
                if isinstance(outputs_raw, tuple):
                    if not outputs_raw: # Empty tuple
                        print("Warning: Compiled model returned empty tuple. Skipping.")
                        continue
                    outputs = outputs_raw[0] # Assume first element is the main output
                else:
                    outputs = outputs_raw # Assume it's the tensor directly
                # --- End tuple handling ---

                # Check if outputs is a tensor before accessing shape
                if not isinstance(outputs, torch.Tensor):
                    print(f"Warning: Model output after potential tuple handling is not a tensor ({type(outputs)}). Skipping.")
                    continue

                if outputs.shape != masks.shape: raise ValueError(f"Shape mismatch! Output: {outputs.shape}, Mask: {masks.shape}")
                loss=bce_dice_hybrid_loss(outputs, masks, ALPHA_BCE, BETA_DICE_BG, GAMMA_DICE_FG) # Use new loss

            if torch.isnan(loss) or torch.isinf(loss):
                nan_detected = True; print(f"\n[Warning] NaN/Inf loss in validation batch {batch_idx}!");
                continue

            batch_loss = loss.item(); running_loss += batch_loss * current_batch_size; num_samples_processed += current_batch_size
            predicted_class = torch.argmax(torch.softmax(outputs, dim=1), dim=1).unsqueeze(1)
            true_classes = torch.argmax(masks, dim=1).unsqueeze(1)
            batch_mean_iou = iou_coefficient(predicted_class, true_classes)
            batch_mean_dice = dice_coefficient(predicted_class, true_classes)
            running_iou += batch_mean_iou; running_dice += batch_mean_dice; num_batches_processed += 1

            if num_batches_processed > 0 and num_samples_processed > 0:
                pbar.set_postfix(loss=f'{batch_loss:.4f}', avg_loss=f'{running_loss/num_samples_processed:.4f}', avg_iou=f'{running_iou/num_batches_processed:.4f}', avg_dice=f'{running_dice/num_batches_processed:.4f}')

    if num_samples_processed == 0 or num_batches_processed == 0:
      return 0.0, 0.0, 0.0
    epoch_loss = running_loss / num_samples_processed
    epoch_iou = running_iou / num_batches_processed
    epoch_dice = running_dice / num_batches_processed
    if nan_detected:
      print("\n[Warning] NaN/Inf detected during validation epoch.")
    return epoch_loss, epoch_iou, epoch_dice

print("Training and validation functions defined.")

Training and validation functions defined.


In [13]:
# --- 11. Utility Functions (Keep GPU Clear, Error Analysis, Email, Visualize) ---
print("Defining utility functions...")
def clear_gpu():
    # ... (clear_gpu remains the same) ...
    if torch.cuda.is_available():
      print("Clearing GPU cache...");
      torch.cuda.empty_cache();
      gc.collect();
      print("GPU cache cleared.");
      time.sleep(2)

Defining utility functions...


In [14]:
# --- Update create_email_body to include AUC ---
def create_email_body(checkpoint_path, encoder, architecture):
    body = f'Training {architecture} finished.\n\nCheckpoint Path: {checkpoint_path}\n\n--- ENCODER: {encoder} ---\n'
    return body

def send_email(subject, body, sender, recipients, password):
    # ... (send_email remains the same) ...
    msg = MIMEText(body); msg['Subject'] = subject; msg['From'] = sender; msg['To'] = ', '.join(recipients)
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp_server: smtp_server.login(sender, password); smtp_server.sendmail(sender, recipients, msg.as_string())
        print("Email sent successfully!")
    except Exception as e: print(f"Error sending email: {e}")

In [15]:
import torch
import numpy as np
from tqdm import tqdm
import gc # Import garbage collector

def find_optimal_threshold_memory_efficient(model, dataloader, device, metric='dice', num_steps=50, smooth=1e-6):
    """
    Finds the optimal probability threshold on the validation set using
    a memory-efficient batch-wise accumulation method.

    Args:
        model: The trained model (set to eval mode).
        dataloader: DataLoader for the validation set.
        device: The device to run computations on.
        metric (str): The metric to optimize ('dice' or 'iou'). Default: 'dice'.
        num_steps (int): Number of thresholds to check between 0 and 1. Default: 50.
        smooth (float): Smoothing factor for Dice/IoU calculation. Default: 1e-6.

    Returns:
        tuple: (best_threshold, best_score)
    """
    model.eval()
    thresholds = np.linspace(0.01, 0.99, num_steps) # Avoid 0 and 1 extremes initially
    best_threshold = 0.5 # Default
    best_score = -1.0    # Initialize with a low score

    # Initialize arrays to store accumulated counts for each threshold
    tp_counts = np.zeros(num_steps, dtype=np.int64)
    fp_counts = np.zeros(num_steps, dtype=np.int64)
    fn_counts = np.zeros(num_steps, dtype=np.int64)

    print(f"Evaluating thresholds on validation set (memory efficient)...")
    with torch.no_grad():
        pbar_batch = tqdm(dataloader, desc="Val Batches for Threshold", leave=False)
        for batch_data in pbar_batch:
            if batch_data is None: continue
            images, masks = batch_data
            if images is None: continue

            images = images.to(device, non_blocking=True)

            # Get true class indices (Cancer = 1) on CPU is fine
            true_indices = torch.argmax(masks, dim=1) # B, H, W, on CPU/GPU depending on masks

            with torch.amp.autocast('cuda'):
                outputs_raw = model(images)
                # Handle tuple output
                if isinstance(outputs_raw, tuple): outputs = outputs_raw[0]
                else: outputs = outputs_raw
                if not isinstance(outputs, torch.Tensor): continue

                # Get probabilities for the positive class (Cancer, index 1)
                outputs_prob_cancer = torch.softmax(outputs, dim=1)[:, 1, :, :] # B, H, W

            # Move probabilities and true indices to CPU for numpy operations if they aren't already
            outputs_prob_cancer_cpu = outputs_prob_cancer.cpu()
            true_indices_cpu = true_indices.cpu() # Ensure it's on CPU

            # Accumulate TP, FP, FN for each threshold for this batch
            for i, threshold in enumerate(thresholds):
                # Apply threshold to get binary predictions
                binary_preds = (outputs_prob_cancer_cpu > threshold).int() # B, H, W

                # Calculate TP, FP, FN for the batch
                batch_tp = ((binary_preds == 1) & (true_indices_cpu == 1)).sum().item()
                batch_fp = ((binary_preds == 1) & (true_indices_cpu == 0)).sum().item()
                batch_fn = ((binary_preds == 0) & (true_indices_cpu == 1)).sum().item()

                # Accumulate counts
                tp_counts[i] += batch_tp
                fp_counts[i] += batch_fp
                fn_counts[i] += batch_fn

            # Optional: Clean up GPU memory within the loop if needed
            del images, masks, outputs, outputs_raw, outputs_prob_cancer
            if device == torch.device('cuda'): torch.cuda.empty_cache()


    # Calculate final scores for each threshold using accumulated counts
    print("Calculating final scores per threshold...")
    if metric == 'dice':
        # Dice = 2 * TP / (2 * TP + FP + FN)
        scores = (2. * tp_counts + smooth) / (2. * tp_counts + fp_counts + fn_counts + smooth)
    elif metric == 'iou':
        # IoU = TP / (TP + FP + FN)
        scores = (tp_counts + smooth) / (tp_counts + fp_counts + fn_counts + smooth)
    else:
        raise ValueError("metric must be 'dice' or 'iou'")

    # Find the best score and corresponding threshold
    best_score_idx = np.argmax(scores)
    best_score = scores[best_score_idx]
    best_threshold = thresholds[best_score_idx]

    print(f"Optimal threshold found: {best_threshold:.4f} with {metric.upper()} score: {best_score:.4f}")


    del tp_counts, fp_counts, fn_counts, scores, thresholds
    gc.collect()

    return best_threshold, best_score

In [16]:
# --- 12. Aim & Ngrok Setup ---
print("Setting up Aim repository...")
# --- Aim Setup ---
from aim import Run
import subprocess
import aim
if not os.path.exists(AIM_REPO_PATH):
  print(f"Aim repository not found at {AIM_REPO_PATH}. Initializing...")
  os.makedirs(AIM_REPO_PATH, exist_ok=True)
  try:
    repo = aim.Repo.init(AIM_REPO_PATH);
    print(f"Aim repository initialized at: {repo.path}")
  except Exception as e:
    print(f"An unexpected error occurred during Aim repo setup: {e}")
else:
  print(f"Using existing Aim repository at: {AIM_REPO_PATH}")
print("Aim/Ngrok setup complete.")

Setting up Aim repository...
Using existing Aim repository at: /content/drive/MyDrive/aim_repo_prostate
Aim/Ngrok setup complete.


In [17]:
def save_metadata(optimal_threshold, best_val_score_for_threshold,checkpoint, encoder, architecture, metadata_best_path):
  # --- SAVE THE OPTIMAL THRESHOLD ---
  meta_filename = os.path.join(f'/content/drive/MyDrive/METADATA_CHECKPOINTS',os.path.basename(metadata_best_path))
  meta_filename = meta_filename.replace(".pth", "_meta.json")

  metadata = {
    "optimal_threshold": optimal_threshold,
    "best_validation_DICE": best_val_score_for_threshold,
    "best_model_epoch": checkpoint.get('epoch', '?'),
    "best_model_val_loss": checkpoint.get('best_val_score', '?'),
    "checkpoint_path": metadata_best_path,
    "encoder": encoder,
    "architecture": architecture,
    "loss_alpha_bce": ALPHA_BCE,
    "loss_beta_dice_bg": BETA_DICE_BG,
    "loss_gamma_dice_fg": GAMMA_DICE_FG,
    "Learnin_rate": BASE_LEARNING_RATE,
    "Encoder_LR_Factor": ENCODER_LR_FACTOR,
    "Weight_Decay": WEIGHT_DECAY,
    "Batch_Size": BATCH_SIZE,
    "Num_Epochs": NUM_EPOCHS,
    "Workers": WORKERS,
    "Seed": SEED,
    "Optimizer": "AdamWScheduleFree_DiffLR",
    "Loss": "BCEDiceHybrid",
    "Freeze_Epochs": FREEZE_EPOCHS,
    "DATASET_ZIP_DIR": DATASET_ZIP_DIR,
    "Dropout": DECODER_DROPOUT,
    "patience": PATIENCE,
    }

  try:
      with open(meta_filename, 'w') as f:
          json.dump(metadata, f, indent=4)
      print(f"Saved optimal threshold and metadata to: {meta_filename}")
  except Exception as e:
    print(f"Error saving metadata file {meta_filename}: {e}")
  # --- END SAVE THRESHOLD ---

In [18]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET": # <-- ADD THIS BLOCK
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights="imagenet",
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model



In [19]:
# ==============================================================================
# --- 13. Main Training Loop (WITH DIFFERENTIAL LR) ---
# ==============================================================================
print(f"\n{'='*25} Starting Main Training Process {'='*25}")
print(f"Base LR: {BASE_LEARNING_RATE:.2E}, Enc Factor: {ENCODER_LR_FACTOR}, Weight Decay: {WEIGHT_DECAY:.1E}")

fold_zip_filename = f'MASTER_SET_1.zip'
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)

# --- Extract Dataset ---
print(f"Extracting Fold...")
if not os.path.exists(fold_zip_path):
  print(f"Zip not found: {fold_zip_path}. Skip.");
try:
    if os.path.exists(base_data_dir):
      shutil.rmtree(base_data_dir)
    os.makedirs(base_data_dir, exist_ok=True);

    with zipfile.ZipFile(fold_zip_path,'r') as z:
      z.extractall(base_data_dir)

    print("Extracted. Verifying...");

    if not os.path.isdir(train_cancer_image_dir) or not os.listdir(train_cancer_image_dir):
      raise RuntimeError("Verify failed")

    print("Verified.")

except Exception as e:
  print(f"Extract Err: {e}. Skip.");
  clear_gpu();


========================= Starting Main Training Process =========================
Base LR: 1.00E-05, Enc Factor: 0.1, Weight Decay: 1.0E-05
Extracting Fold...
Extracted. Verifying...
Verified.


In [20]:
# --- DataLoaders ---
print("\nCreating DataLoaders...")
try:
    train_ds=ProstateCancerDataset(train_cancer_image_dir,train_cancer_mask_dir,train_not_cancer_image_dir,train_not_cancer_mask_dir);
    val_ds=ProstateCancerDataset(val_cancer_image_dir,val_cancer_mask_dir,val_not_cancer_image_dir,val_not_cancer_mask_dir);

    print(f"\n--- DS Lengths Fold ---\n Train:{len(train_ds)}, Val:{len(val_ds)}\n{'-'*30}")

    def collate_fn(batch):
      batch=list(filter(lambda x:x is not None and x[0] is not None,batch));
      return torch.utils.data.dataloader.default_collate(batch) if batch else None

    train_loader=DataLoader(train_ds,BATCH_SIZE,shuffle=True,num_workers=WORKERS,pin_memory=True,drop_last=True,persistent_workers=WORKERS>0,prefetch_factor=2 if WORKERS>0 else None,collate_fn=collate_fn)
    val_loader=DataLoader(val_ds,BATCH_SIZE,shuffle=False,num_workers=WORKERS,pin_memory=True,persistent_workers=WORKERS>0,prefetch_factor=2 if WORKERS>0 else None,collate_fn=collate_fn)

    print("DataLoaders created.")

except Exception as e:
  print(f"DataLoader Err: {e}. Skip.");
  clear_gpu();


Creating DataLoaders...

--- DS Lengths Fold ---
 Train:37158, Val:8771
------------------------------
DataLoaders created.


In [21]:
#LIST_ARCH = ['SWIN', 'DEEPLABV3PLUS', 'INCEPTIONRESNETV2', 'UNET++', 'FPN', 'SEGFORMER', 'MANET', 'DPT']
#LIST_ENCODER = ['swin_large_patch4_window7_224','resnet152','inceptionresnetv2','resnet152','resnet152','mit_b5','resnet152', 'tu-vit_base_patch16_224.augreg_in21k']

# The new, refactored lists
# LIST_ARCH = ['SWIN', 'DEEPLABV3PLUS', 'UNET++', 'FPN', 'SEGFORMER', 'MANET', 'DPT', 'UPerNet']

# LIST_ENCODER = [
#     'tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k', # For Unet (acting as our new Swin model)
#     'tu-resnest101e',                                   # For DEEPLABV3PLUS
#     'efficientnet-b7',                                    # For UNET++
#     'senet154',                                           # For FPN
#     'mit_b5',                                             # For SEGFORMER
#     'resnet152',                                          # For MANET
#     'tu-vit_large_patch16_224.augreg_in21k_ft_in1k',     # For DPT
#     'tu-hiera_large_224'                                # For UPerNet
# ]

LIST_ARCH = ['SWIN']

LIST_ENCODER = ['tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k']

for architecture, encoder in zip (LIST_ARCH, LIST_ENCODER):
  # --- Init Aim Run ---
  experiment_name = f"{architecture}_{encoder}_{get_formatted_datetime_string()}"
  print(f"Init Aim: {experiment_name}")
  run = None
  try:
      run = Run(experiment=experiment_name, repo=AIM_REPO_PATH)
      run["hparams"] = {
          "base_learning_rate": BASE_LEARNING_RATE, "encoder_lr_factor": ENCODER_LR_FACTOR, "weight_decay": WEIGHT_DECAY,
          "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,"workers": WORKERS, "seed": SEED,"label": experiment_name,
          "optimizer": "AdamWScheduleFree_DiffLR","loss": "BCEDiceHybrid","model": f"{architecture}_{encoder}", # Use SMP name
          "encoder_weights": "IMAGENET", "patience": PATIENCE, "freeze_epochs": FREEZE_EPOCHS,
          "train_len":len(train_ds),"val_len":len(val_ds),
          "loss_alpha_bce": ALPHA_BCE, "loss_beta_dice_bg": BETA_DICE_BG, "loss_gamma_dice_fg": GAMMA_DICE_FG,
      }
      print("Aim run initialized.")
  except Exception as e:
    print(f"Aim Init Err: {e}.")

  print(f"Architecture: {architecture}")
  # --- Initialize Model, Optimizer, Early Stopping ---
  print("Initializing SMP Model, Optimizer, ES...")

  # --- Instantiate SMP Model ---
  try:
    model = get_model(architecture = architecture, encoder = encoder, validation = False)
    model.to(device);
  except Exception as e:
    print(f"Model init err: {e}")
    raise ValueError(f"Unknown architecture: {architecture}")

  if int(FREEZE_EPOCHS)!=0:
    model._encoder_frozen = True
  else:
    model._encoder_frozen = False

  # --- Optionally print model summary ---
  try:
        summary(model, input_size=(BATCH_SIZE, 3, 224, 224), device=str(device))
  except Exception as e:
        print(f"Could not print model summary: {e}")

  # --- Compile Model ---
  try:
    model = torch.compile(model);
    print("Model compiled.")
  except Exception as e:
    print(f"Compile failed: {e}.")

  print(f"Model->{device}")

  if int(FREEZE_EPOCHS)!=0:
    # --- Initial Optimizer Setup (Encoder Frozen using SMP structure) ---
    print("Setting up initial optimizer (encoder frozen)...")
    # Freeze encoder initially
    for param in model.encoder.parameters():
        param.requires_grad = False

    # Identify trainable parameters (decoder + segmentation head)
    trainable_params_frozen_phase = [
        p for p in model.decoder.parameters() if p.requires_grad
    ] + [
        p for p in model.segmentation_head.parameters() if p.requires_grad
    ]

    if not trainable_params_frozen_phase: print("ERROR: No trainable decoder/head parameters!"); continue

    optimizer = schedulefree.AdamWScheduleFree(
        trainable_params_frozen_phase,
        lr=BASE_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    print(f"Optimizer initialized for frozen phase with {len(trainable_params_frozen_phase)} params.")
  else:
    optimizer = schedulefree.AdamWScheduleFree(
    model.parameters(),
    lr=BASE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY)


  early_stopping = EarlyStopping(patience=int(PATIENCE), verbose=True, delta=0.0001, mode='max') # Monitor val_loss
  best_model_path = f'/content/best_model_fold_{experiment_name}.pth'
  metadata_best_path = None # Initialize metadata_best_path

  print(f"Starting Training For {architecture}...")

  # --- Training Loop for Current Fold ---

  training_successful = True
  for epoch in range(NUM_EPOCHS):
      current_epoch_num = epoch + 1
      epoch_start = time.time()

      if int(FREEZE_EPOCHS)!=0:
        # --- Check and Re-initialize Optimizer for Unfreezing ---
        if epoch == FREEZE_EPOCHS:
            print(f"\nEpoch {current_epoch_num}: Unfreezing encoder & re-initializing optimizer with differential LR...")
            # Unfreeze encoder parameters
            for param in model.encoder.parameters():
                param.requires_grad = True
            model._encoder_frozen = False

            # Define parameter groups for the unfrozen state using SMP structure
            decoder_params = [p for p in model.decoder.parameters() if p.requires_grad]
            head_params = [p for p in model.segmentation_head.parameters() if p.requires_grad]
            encoder_params = [p for p in model.encoder.parameters() if p.requires_grad] # Should now require grad

            if not encoder_params: print("ERROR: No trainable encoder parameters after unfreezing!"); training_successful = False; break
            if not (decoder_params + head_params): print("ERROR: No trainable decoder/head parameters after unfreezing!"); training_successful = False; break

            # Create the new optimizer with differential learning rates
            optimizer = schedulefree.AdamWScheduleFree([
                {'params': decoder_params + head_params, 'lr': BASE_LEARNING_RATE}, # Base LR for decoder + head
                {'params': encoder_params, 'lr': BASE_LEARNING_RATE * ENCODER_LR_FACTOR} # Scaled LR for encoder
            ], lr=BASE_LEARNING_RATE, weight_decay=WEIGHT_DECAY) # Base LR here is default

            print(f"Optimizer re-initialized with {len(decoder_params)+len(head_params)} decoder/head params (LR={BASE_LEARNING_RATE:.2E}) "
                  f"and {len(encoder_params)} encoder params (LR={BASE_LEARNING_RATE * ENCODER_LR_FACTOR:.2E}).")

      # --- Train & Validate ---

      try:
        train_loss,train_iou,train_dice=train_model(model,optimizer,train_loader,device,epoch,FREEZE_EPOCHS)
      except Exception as e:
        print(f"\nTrain Err E{current_epoch_num}:{e}");
        training_successful=False;
        break

      try:
        val_loss,val_iou,val_dice=validate_model(model,optimizer,val_loader,device)
      except Exception as e:
        print(f"\nVal Err E{current_epoch_num}:{e}");
        training_successful=False;
        break

      # --- Logging ---
      epoch_dur=time.time()-epoch_start; mins,secs=divmod(epoch_dur,60)
      print(f"\nE{current_epoch_num}/{NUM_EPOCHS} [{int(mins):02d}m{int(secs):02d}s] Tr L:{train_loss:.4f} IoU:{train_iou:.4f} Di:{train_dice:.4f} | Val L:{val_loss:.4f} IoU:{val_iou:.4f} Di:{val_dice:.4f}")
      if run:
        try:
              run.track(train_loss,'loss',epoch=current_epoch_num,context={"subset":"train"});
              run.track(train_iou,'iou',epoch=current_epoch_num,context={"subset":"train"});
              run.track(train_dice,'dice',epoch=current_epoch_num,context={"subset":"train"})
              run.track(val_loss,'loss',epoch=current_epoch_num,context={"subset":"val"});
              run.track(val_iou,'iou',epoch=current_epoch_num,context={"subset":"val"});
              run.track(val_dice,'dice',epoch=current_epoch_num,context={"subset":"val"})
        except Exception as e: print(f"Aim Log Err: {e}")

      # --- Checkpoint & Early Stopping ---
      chkpt_dir=f'/content/drive/MyDrive/Checkpoints';
      os.makedirs(chkpt_dir,exist_ok=True);
      periodic_chkpt=os.path.join(chkpt_dir, f'{experiment_name}_E{current_epoch_num}_VLoss_{val_loss:.4f}.pth') # Use VLoss in name
      if metadata_best_path is None:
        metadata_best_path = periodic_chkpt

      try:
        improvement_detected = early_stopping(val_dice, model, best_model_path, periodic_chkpt, optimizer, current_epoch_num, val_loss, val_iou) # Monitor val_loss
        if improvement_detected:
          metadata_best_path = periodic_chkpt
      except Exception as e:
        print(f"ES/Save Err: {e}")

      if not UNLEASHED and early_stopping.early_stop:
        print(f"Early stopping E{current_epoch_num}.");
        break


  # --- Post-Training for Fold ---
  if not training_successful:
    print(f"Train loop stopped early for {architecture}.")
  else:
    print(f"\nTrain loop finished for {architecture}.");
    clear_gpu()

  # --- Load Best Model (based on Val Loss) ---
  print("Loading best model for threshold tuning and testing...")
  optimal_threshold = 0.5 # Default threshold
  best_val_score_for_threshold = -1.0 # Reset score for this fold
  test_metrics = None # Initialize test_metrics

  try:
    if os.path.exists(best_model_path):
        chkpt=torch.load(best_model_path,map_location=device);

        # --- Instantiate SMP Model ---
        try:
          model_test = get_model(architecture = architecture, encoder = encoder, validation = True)
          model_test.to(device)
        except Exception as e:
          print(f"Model init err: {e}")
          raise ValueError(f"Unknown architecture: {architecture}")

        print("Loading state dict...");
        model_test.load_state_dict(chkpt['model_state_dict'],strict=False);
        print("Loaded.");
        is_comp=chkpt.get('is_compiled',False)

        if is_comp:
          print("Compiling test model...");
          try:
            model_test=torch.compile(model_test);
            print("Compiled.")
          except Exception as e:
            print(f"Compile fail: {e}")

        # --- Find Optimal Threshold on Validation Set ---
        optimal_threshold, best_val_score_for_threshold = find_optimal_threshold_memory_efficient(
            model=model_test,
            dataloader=val_loader, # Use validation loader
            device=device,
            metric='dice', # Choose 'dice' or 'iou'
            num_steps=50 # Adjust number of thresholds to check (e.g., 50-100)
        )

        save_metadata(optimal_threshold, best_val_score_for_threshold, chkpt, encoder, architecture, metadata_best_path)

        print("Emailing...");
        subject = f"Training {architecture} finished"
        body=create_email_body(metadata_best_path, encoder, architecture); # Use metadata_best_path here
        send_email(f"Finished: {experiment_name}",body,sender,recipients,password)
    else:
      print(f"Best model not found: {best_model_path}. Skip test.")

  except Exception as e:
    print(f"Test/Visu Err: {e}");
  import traceback;
  traceback.print_exc()

  if run:
    run.close(); print("Aim run closed.")

  clear_gpu();
  print(f"\n{'='*20} Finished Fold {architecture} {'='*20}");
  time.sleep(3)


# --- Final Cleanup ---
print("\nAll folds processed.")

Init Aim: SWIN_tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k_14_10_2025_02_37_47
Aim run initialized.
Architecture: SWIN
Initializing SMP Model, Optimizer, ES...


model.safetensors:   0%|          | 0.00/788M [00:00<?, ?B/s]

Model compiled.
Model->cuda
Starting Training For SWIN...


Train E1:   0%|          | 0/2322 [00:00<?, ?it/s]W1014 02:39:00.013000 5356 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode



E1/100 39s] Tr L:0.4810 IoU:0.4940 Di:0.5081 | Val L:0.4101 IoU:0.9098 Di:0.9109
Initial best score (max mode): 0.910929. Saving model ...


...


E6/100 02s] Tr L:0.2740 IoU:0.9844 Di:0.9846 | Val L:0.4302 IoU:0.9076 Di:0.9085
EarlyStopping counter: 4 out of 5 (Best max score: 0.929338)



E7/100 00s] Tr L:0.2657 IoU:0.9924 Di:0.9925 | Val L:0.4629 IoU:0.9040 Di:0.9048
EarlyStopping counter: 5 out of 5 (Best max score: 0.929338)
Early stopping E7.

Train loop finished for SWIN.
Clearing GPU cache...
GPU cache cleared.
Loading best model for threshold tuning and testing...
Loading state dict...
Loaded.
Compiling test model...
Compiled.
Evaluating thresholds on validation set (memory efficient)...


Calculating final scores per threshold...
Optimal threshold found: 0.1700 with DICE score: 0.9436
Saved optimal threshold and metadata to: /content/drive/MyDrive/METADATA_CHECKPOINTS/SWIN_tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k_14_10_2025_02_37_47_E2_VLoss_0.3892_meta.json
Emailing...
Email sent successfully!
Aim run closed.
Clearing GPU cache...


NoneType: None


GPU cache cleared.

==================== Finished Fold SWIN ====================

All folds processed.
